In [1]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

In [2]:
from cyclical_MCMC import load_data

# Load Data

In [3]:
Y, dY, dY_pd, Y_indx, _ = load_data('data/SP500.csv')

Data from 2016-09-19 00:00:00 to 2026-09-17 00:00:00


# Simulating Heston

In [5]:
seed = 42
np.random.seed(seed)

In [20]:
def draw_dWs(rho, Delta, N=1):

    # draw normals
    N1 = np.random.normal(size=(N))
    N2 = np.random.normal(size=(N))

    # setup BMs
    dW1 = np.sqrt(Delta) * N1
    dW2 = rho * dW1 + np.sqrt(1-rho**2) * np.sqrt(Delta) * N2

    return dW1, dW2

def full_trunc_euler(mu, kappa, theta, sigma_sq_V, rho, Delta, N=1, T=1, burnin=0, Y0=None, V0=None):

    # allocate
    V = np.zeros((T+burnin, N)) + np.nan
    S = np.zeros((T+burnin, N)) + np.nan
    Y = np.zeros((T+burnin, N)) + np.nan
    dY = np.zeros((T-1+burnin, N)) + np.nan

    # initialize logged stock price at 0
    if Y0 is None:
        Yt = np.log(np.ones(N)) # (N,)

    else:
        Yt = Y0

    # specified initial variance
    if V0 is None:
        Vt = np.ones(N)*theta # (N,)
    else:
        Vt = V0

    for i in range(T+burnin):

        # draw Brownian motions
        dW1, dW2 = draw_dWs(rho, Delta, N=N)

        # simulate logged stock price
        Vtpos = np.maximum(Vt, 0.00)
        Yt = Yt + mu * Delta + np.sqrt(Vtpos) * dW1

        # simulate instantenous variance
        Vt = Vt + kappa * (theta - Vtpos) * Delta + np.sqrt(sigma_sq_V*Vtpos) * dW2

        # save
        V[i] = Vt
        Y[i] = Yt

        if i > 0:
            dY[i-1] = Y[i] - Y[i-1]

    return S[burnin:], V[burnin:], Y[burnin:], dY[burnin:]

In [21]:
years = 10
days_pr_year = 252
T = years*days_pr_year # total number of days
N = 10000
Delta = 1

mu = dY.mean()
theta = dY.var()
kappa = 0.02
sigma_sq_V = 0.225**2
rho = -0.8

print(f'Feller is fulfilled: {2*kappa*theta > sigma_sq_V}')
print(f'2 kappa theta = {2*kappa*theta}')
print(f'sigma_V^2 = {sigma_sq_V}')

Feller is fulfilled: True
2 kappa theta = 0.05209619676004937
sigma_V^2 = 0.050625


In [54]:
from ll import obs_ll_t

In [76]:
# sample V0 given par
T_sample0 = 100
_, V0_sims, _, _ = full_trunc_euler(mu, kappa, theta, sigma_sq_V, rho, Delta, N=N, T=T_sample0, burnin=0)
V0_sim = V0_sims[-1]

In [92]:
# push forward particles
V0_sim_pos = np.maximum(V0_sim, 1e-6)
V1_sim = V0_sim + kappa * (theta - V0_sim_pos) * Delta + np.sqrt(sigma_sq_V*V0_sim_pos) * np.random.normal(size=(N,))

# compute weights
V1_sim_pos = np.maximum(V1_sim, 1e-6)
log_w1 = obs_ll_t(mu, kappa, kappa*theta, sigma_sq_V, rho, Y[0], Y[1], V0_sim_pos, V1_sim_pos, Delta)

# normalize weights
m = log_w1.max()
w1 = np.exp(log_w1 - m)
w1 = w1/np.sum(w1)
V1_sim_resampled = np.random.choice(V1_sim_pos, p=w1, size=(N,))